# Ejemplo de procesos de ciencia de datos

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gf0657-programacionsig/2026-ii/blob/main/contenidos/i-introduccion-ciencia-datos-programacion/02-ejemplo-procesos-ciencia-datos.ipynb)

## Introducción

Este cuaderno de notas ilustra los procesos de ciencia de datos descritos en la lección de [introducción a la ciencia de datos](01-introduccion-ciencia-datos.md) —importación, estructuración, transformación y visualización—, aplicados a registros de presencia de la lapa verde (*Ara ambiguus*), especie en peligro crítico de extinción. Se utilizan las bibliotecas:

- [Pygbif](https://pygbif.readthedocs.io/): acceso a los datos de [GBIF](https://www.gbif.org/) mediante su interfaz de programación de aplicaciones (API).
- [pandas](https://pandas.pydata.org/): estructuración y transformación de datos tabulares.
- [Plotly](https://plotly.com/python/): gráficos estadísticos interactivos.
- [Folium](https://python-visualization.github.io/folium/): mapas interactivos.

La insignia del inicio abre este cuaderno en [Google Colab](https://colab.research.google.com/), donde puede ejecutarse sin instalar nada en la computadora local. Para conservar los cambios, use *File > Save a copy in Drive*.

## Instalación y carga de bibliotecas

In [1]:
# Instalación de pygbif, necesaria en Google Colab
# (el ambiente conda del curso ya la incluye)
%pip install pygbif --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Carga de bibliotecas
from pygbif import occurrences
import pandas as pd
import plotly.express as px
import folium

# Función auxiliar usada en este cuaderno en lugar de fig.show().
# Despliega el gráfico mediante display(fig), lo que produce una salida
# que el sitio web del curso, Jupyter y Colab muestran de forma interactiva.
from IPython.display import display


def mostrar(fig):
    """Despliega un gráfico de plotly de forma compatible con el sitio del curso."""
    display(fig)

## Parámetros generales

In [3]:
# Nombre científico de la especie
especie = "Ara ambiguus"

## Importación

Los datos de presencia se obtienen por medio de solicitudes (*requests*) al API de GBIF mediante la biblioteca Pygbif. Como el API entrega los resultados en "páginas" de 300 registros como máximo, se itera hasta recuperarlos todos.

In [4]:
limite = 300  # Límite de registros por solicitud
offset = 0  # Desplazamiento para iniciar la solicitud
registros_acumulados = []  # Lista para acumular los resultados

while True:
    # Solicitud
    res = occurrences.search(
        scientificName=especie,
        hasCoordinate=True,
        hasGeospatialIssue=False,
        limit=limite,
        offset=offset
    )

    # Extraer resultados
    registros = res.get("results", [])

    # Si ya no hay resultados, se detiene el ciclo
    if not registros:
        break

    # Agregar registros nuevos al acumulado
    registros_acumulados.extend(registros)

    # Actualizar el offset para la siguiente "página"
    offset += limite

## Estructuración

In [5]:
# Convertir todo a un DataFrame de pandas
presencia = pd.DataFrame(registros_acumulados)

In [6]:
# Cantidad de registros de presencia recuperados
print(len(presencia))

1525


In [7]:
# Muestra de los registros recuperados
presencia[['species', 'basisOfRecord', 'countryCode', 'locality', 'decimalLongitude', 'decimalLatitude', 'eventDate', 'year']].sample(10)

,species,basisOfRecord,countryCode,locality,decimalLongitude,decimalLatitude,eventDate,year
639,Ara ambiguus,HUMAN_OBSERVATION,CR,NaN,-83.421125,10.526342,2023-04-18T04:43,2023.0
1502,Ara ambiguus,PRESERVED_SPECIMEN,CO,Rio Jurado,-77.757940,7.106113,1940-09-15,1940.0
949,Ara ambiguus,HUMAN_OBSERVATION,CR,NaN,-84.235533,10.691017,2020-03-07T09:00,2020.0
90,Ara ambiguus,HUMAN_OBSERVATION,CR,NaN,-83.570514,10.438132,2026-03-28T13:24,2026.0
366,Ara ambiguus,HUMAN_OBSERVATION,CR,NaN,-84.180465,10.509413,2024-01-15T15:20:45,2024.0
346,Ara ambiguus,HUMAN_OBSERVATION,CR,NaN,-84.255918,10.603149,2025-11-21T09:20,2025.0
361,Ara ambiguus,HUMAN_OBSERVATION,NI,NaN,-86.037344,11.868790,2025-12-13T08:53:38,2025.0
1520,Ara ambiguus,HUMAN_OBSERVATION,CR,Agrimaga,-83.621924,10.213607,NaN,NaN
807,Ara ambiguus,HUMAN_OBSERVATION,CR,La Selva Reserve,-84.005989,10.431014,2022-07-18,2022.0
1192,Ara ambiguus,HUMAN_OBSERVATION,CR,NaN,-83.655861,10.466838,2017-04-12T06:31,2017.0


## Transformación

In [8]:
# Año inicial
anio_inicial = 2010

# Filtro por año inicial
presencia = presencia[presencia['year'] >= anio_inicial]

In [9]:
# Cantidad de registros de presencia después del filtro
print(len(presencia))

1442


## Visualización

### Gráfico de barras: registros de presencia por país

In [10]:
# Agrupar por código de país
registros_x_pais = (
    presencia
    .groupby('countryCode', dropna=True)
    .size()
    .reset_index(name='count')
)

# Ordenar descendentemente por frecuencia
registros_x_pais = registros_x_pais.sort_values('count', ascending=False)

# Crear gráfico de barras
fig = px.bar(
    registros_x_pais,
    x='countryCode',
    y='count',
    title='Cantidad de registros de presencia por país',
    labels={'countryCode': 'País', 'count': 'Cantidad de registros de presencia'},
    text='count'
)

# Personalizar tooltip (hover) y estilo de las barras
fig.update_traces(
    hovertemplate='País: %{x}<br>Cantidad de registros de presencia: %{y}',
    marker_color='rgb(55, 83, 109)'
)

# Opciones de diseño
fig.update_layout(
    template='plotly_white',
    xaxis={'type': 'category'},
    xaxis_title='País',
    yaxis_title='Cantidad de registros de presencia',
    showlegend=False,
    annotations=[
        dict(
            text='Fuente: GBIF',
            x=1, y=1,
            xref='paper', yref='paper',
            xanchor='right', yanchor='bottom',
            showarrow=False
        )
    ]
)

# Mostrar la figura
mostrar(fig)

### Gráfico de líneas: registros de presencia por año

In [11]:
# Agrupar por año
registros_x_anio = (
    presencia
    .groupby('year', dropna=True)
    .size()
    .reset_index(name='n')
)

# Crear gráfico de líneas
fig = px.line(
    registros_x_anio,
    x='year',
    y='n',
    labels={'year': 'Año', 'n': 'Cantidad de registros de presencia'},
    title='Cantidad de registros de presencia por año'
)

# Personalizar tooltip (hover) y estilo de la línea
fig.update_traces(
    mode='lines+markers',
    hovertemplate=(
        "Año: %{x}<br>"
        "Cantidad de registros: %{y}"
    )
)

# Opciones de diseño
fig.update_layout(
    annotations=[
        dict(
            text='Fuente: GBIF',
            x=1, y=0,
            xref='paper', yref='paper',
            xanchor='right', yanchor='bottom',
            showarrow=False
        )
    ],
    xaxis_title='Año',
    yaxis_title='Cantidad de registros de presencia'
)

# Mostrar la figura
mostrar(fig)

### Mapa: ubicación de los registros de presencia

In [12]:
# Crear el mapa base, centrado en el área de distribución de la especie
m = folium.Map(
    location=[8.5, -81],
    zoom_start=5,
    tiles='OpenStreetMap',
    name='Mapa general'
)

# Agregar capa de teselas de Esri WorldImagery (imágenes satelitales)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Imágenes satelitales',
    overlay=False,
    control=True
).add_to(m)

# Agregar capa de teselas de CartoDB positron (mapa en blanco)
folium.TileLayer(
    'CartoDB positron',
    name='Mapa en blanco',
    overlay=False,
    control=True
).add_to(m)

# Crear un FeatureGroup para los registros de presencia
fg_presencia = folium.FeatureGroup(name='Registros de presencia')

# Iterar sobre el DataFrame y agregar cada punto
for i, row in presencia.iterrows():
    # Extraer coordenadas
    lat = row['decimalLatitude']
    lon = row['decimalLongitude']

    # Verificar que lat y lon sean válidos
    if pd.notnull(lat) and pd.notnull(lon):
        # Construir popup en HTML
        popup_html = (
            f"<strong>País: </strong>{row.get('country','')}<br/>"
            f"<strong>Localidad: </strong>{row.get('locality','')}<br/>"
            f"<strong>Fecha: </strong>{row.get('eventDate','')}<br/>"
            f"<strong>Fuente: </strong>{row.get('institutionCode','')}<br/>"
            f"<a href='{row.get('occurrenceID','#')}' target='_blank'>Más información</a>"
        )

        # Agregar marcador circular
        folium.CircleMarker(
            location=[lat, lon],
            radius=3,
            fill=True,
            fill_color='red',
            fill_opacity=1,
            stroke=False,
            popup=popup_html
        ).add_to(fg_presencia)

# Agregar el FeatureGroup al mapa
fg_presencia.add_to(m)

# Agregar el control de capas
folium.LayerControl().add_to(m)

# Mostrar el mapa
m

## Referencias bibliográficas

GBIF. (s. f.). *Technical documentation: Occurrence API*. https://techdocs.gbif.org/
\
\
pygbif. (s. f.). *pygbif: GBIF Python client*. https://pygbif.readthedocs.io/